In [1]:
pip install catboost xgboost scikit-learn==1.5.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 19.2 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [2]:
pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 47.1 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.14.1
    Uninstalling scipy-1.14.1:
      Successfully uninstalled scipy-1.14.1


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
import gensim
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

import torch
import math
from sklearn.cluster import DBSCAN
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback, RobertaModel, RobertaTokenizer

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
leetcode_questions_df = pd.read_csv('/content/drive/MyDrive/thesis/leetcode/part4 feature-engineering/leetcode_questions_df.csv')

leetcode_questions_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61834 entries, 0 to 61833
Data columns (total 31 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   username                            61834 non-null  object 
 1   country                             61834 non-null  object 
 2   contest_url                         61834 non-null  object 
 3   num_of_contest                      61834 non-null  int64  
 4   is_weekly                           61834 non-null  bool   
 5   rank                                61834 non-null  int64  
 6   score                               61834 non-null  int64  
 7   question_number                     61834 non-null  int64  
 8   question_language                   61834 non-null  object 
 9   question_code                       61834 non-null  object 
 10  number_of_lines                     61834 non-null  int64  
 11  names_set                           61834

In [6]:
leetcode_questions_df = leetcode_questions_df[leetcode_questions_df['question_language'] == 'java']
leetcode_questions_df = leetcode_questions_df[leetcode_questions_df['question_number'] > 2]
leetcode_questions_df = leetcode_questions_df.drop_duplicates(subset=['question_code'])
leetcode_questions_df = leetcode_questions_df[leetcode_questions_df['country'].isin(leetcode_questions_df['country'].value_counts().head(2).index)]

leetcode_questions_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2101 entries, 524 to 61818
Data columns (total 31 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   username                            2101 non-null   object 
 1   country                             2101 non-null   object 
 2   contest_url                         2101 non-null   object 
 3   num_of_contest                      2101 non-null   int64  
 4   is_weekly                           2101 non-null   bool   
 5   rank                                2101 non-null   int64  
 6   score                               2101 non-null   int64  
 7   question_number                     2101 non-null   int64  
 8   question_language                   2101 non-null   object 
 9   question_code                       2101 non-null   object 
 10  number_of_lines                     2101 non-null   int64  
 11  names_set                           2101 non-

In [7]:
model_name = "neulab/codebert-java"
tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaModel.from_pretrained(model_name)

code_snippets = leetcode_questions_df.question_code.tolist()

# Step 1: Encode the code snippets using CodeBERT
def get_embeddings(code_snippet):
    inputs = tokenizer(code_snippet, return_tensors='pt', truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    # Use the last hidden state of the [CLS] token as the embedding
    return outputs.last_hidden_state[:, 0, :].numpy()

# Get embeddings for all code snippets
embeddings = np.vstack([get_embeddings(snippet) for snippet in code_snippets])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.54k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/696 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at neulab/codebert-java and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [8]:
min_samples = 10 ** (math.floor(math.log10(len(code_snippets))) - 1)

min_samples

100

In [9]:
# Step 2: Apply DBSCAN for clustering and outlier detection
dbscan = DBSCAN(eps=0.1, min_samples=min_samples, metric='cosine', n_jobs=-1)
db_labels = dbscan.fit_predict(embeddings)

# Step 3: Identify and handle outliers
outliers = np.where(db_labels == -1)[0]

# Output some statistics
print(f'Removed {len(outliers)} outliers.')
print(f'Retained {len(db_labels) - len(outliers)} code snippets.')

Removed 53 outliers.
Retained 2048 code snippets.


In [10]:
# Remove outliers from the DataFrame
leetcode_questions_df.reset_index(drop=True, inplace=True)
leetcode_questions_df = leetcode_questions_df[~leetcode_questions_df.index.isin(outliers)]

leetcode_questions_df.country.value_counts()

,count
country,
India,1344
United States,704


In [11]:
leetcode_questions_df.drop(['num_of_contest','is_weekly','rank','score','global_rank_percentile','num_contests_participated','question_number'],axis=1, inplace=True)

In [12]:
X=leetcode_questions_df.drop('country',axis=1)
Y=leetcode_questions_df.country

In [13]:
X_nontext=X.select_dtypes(exclude=['object'])
X_nontext.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2048 entries, 0 to 2100
Data columns (total 18 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   number_of_lines                     2048 non-null   int64  
 1   token_count                         2048 non-null   int64  
 2   variables_count                     2048 non-null   int64  
 3   function_count                      2048 non-null   int64  
 4   loop_count                          2048 non-null   int64  
 5   condition_count                     2048 non-null   int64  
 6   single_line_comment_density         2048 non-null   float64
 7   multiline_comment_density           2048 non-null   float64
 8   function_density                    2048 non-null   float64
 9   loop_density                        2048 non-null   float64
 10  condition_density                   2048 non-null   float64
 11  comment_tokens_density              2048 non-nul

In [14]:
X_train_nontext, X_test_nontext, y_train, y_test = train_test_split(X_nontext, Y, test_size=0.2, random_state=0,stratify=Y)

In [15]:
X_train_text, X_test_text, Y_train, y_test = train_test_split(X.question_code, Y, test_size=0.2, random_state=0,stratify=Y)

In [16]:
def read_corpus(text, tokens_only=False):
    for i, line in enumerate(text):
        tokens = gensim.utils.simple_preprocess(line)
        if tokens_only:
            yield tokens
        else:
        # For training data, add tags
            yield gensim.models.doc2vec.TaggedDocument(tokens, [i])

train_corpus = list(read_corpus(X_train_text))
test_corpus = list(read_corpus(X_test_text, tokens_only=True))

In [17]:
model = gensim.models.doc2vec.Doc2Vec(vector_size=300, min_count=2)
model.build_vocab(train_corpus)
model.train(train_corpus, total_examples=model.corpus_count, epochs=55)

In [18]:
vectors = [model.infer_vector(train_corpus[doc_id].words) for doc_id in range(len(train_corpus))]
X_train_doc2vec = np.vstack(vectors)

test_vectors = [model.infer_vector(test_corpus[doc_id]) for doc_id in range(len(test_corpus))]
X_test_doc2vec = np.vstack(test_vectors)

X_train_doc2vec.shape , X_test_doc2vec.shape

((1638, 300), (410, 300))

In [19]:
X_train_combined=pd.concat([pd.DataFrame(X_train_doc2vec, columns=['doc2vec_'+str(i) for i in range(300)], index=X_train_nontext.index),X_train_nontext],axis=1)

X_test_combined=pd.concat([pd.DataFrame(X_test_doc2vec, columns=['doc2vec_'+str(i) for i in range(300)], index=X_test_nontext.index),X_test_nontext],axis=1)

# Cat Boost

In [20]:
baseline_model = Pipeline([('scaler',StandardScaler()),
                           ('classifier',CatBoostClassifier(iterations=10))])

In [21]:
scores = cross_val_score(baseline_model, X_train_nontext, y_train, cv=5)
print("baseline model score: ",np.mean(scores))

Learning rate set to 0.5
0:	learn: 0.6416583	total: 48.5ms	remaining: 436ms
1:	learn: 0.6158049	total: 49.5ms	remaining: 198ms
2:	learn: 0.5989626	total: 50.6ms	remaining: 118ms
3:	learn: 0.5860969	total: 51.6ms	remaining: 77.4ms
4:	learn: 0.5805431	total: 52.6ms	remaining: 52.6ms
5:	learn: 0.5762676	total: 53.6ms	remaining: 35.7ms
6:	learn: 0.5689045	total: 54.6ms	remaining: 23.4ms
7:	learn: 0.5600293	total: 55.6ms	remaining: 13.9ms
8:	learn: 0.5527823	total: 56.7ms	remaining: 6.29ms
9:	learn: 0.5466420	total: 57.7ms	remaining: 0us
Learning rate set to 0.5
0:	learn: 0.6388218	total: 1.56ms	remaining: 14.1ms
1:	learn: 0.6123039	total: 2.58ms	remaining: 10.3ms
2:	learn: 0.5896624	total: 3.63ms	remaining: 8.48ms
3:	learn: 0.5802348	total: 4.69ms	remaining: 7.03ms
4:	learn: 0.5733818	total: 5.77ms	remaining: 5.77ms
5:	learn: 0.5616287	total: 6.89ms	remaining: 4.59ms
6:	learn: 0.5475775	total: 7.97ms	remaining: 3.42ms
7:	learn: 0.5432304	total: 9.04ms	remaining: 2.26ms
8:	learn: 0.5412581	

In [22]:
param_grid = {
    'classifier__learning_rate': [0.01, 0.1, 0.3],
    'classifier__l2_leaf_reg': [1, 5 ,10],
}

grid_search = GridSearchCV(estimator=baseline_model, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2, scoring='f1_macro')

grid_search.fit(X_train_nontext, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_cb = grid_search.best_estimator_
y_pred = best_cb.predict(X_test_nontext)
print("Test set F1 score: ", f1_score(y_test, y_pred, average='macro'))

Fitting 5 folds for each of 9 candidates, totalling 45 fits
0:	learn: 0.6512592	total: 1.54ms	remaining: 13.9ms
1:	learn: 0.6262581	total: 2.88ms	remaining: 11.5ms
2:	learn: 0.6057701	total: 4ms	remaining: 9.33ms
3:	learn: 0.5961139	total: 5.06ms	remaining: 7.59ms
4:	learn: 0.5877632	total: 6.12ms	remaining: 6.12ms
5:	learn: 0.5836595	total: 7.12ms	remaining: 4.75ms
6:	learn: 0.5781486	total: 8.16ms	remaining: 3.5ms
7:	learn: 0.5718360	total: 9.24ms	remaining: 2.31ms
8:	learn: 0.5677183	total: 10.2ms	remaining: 1.13ms
9:	learn: 0.5629584	total: 11.3ms	remaining: 0us
Best parameters found:  {'classifier__l2_leaf_reg': 1, 'classifier__learning_rate': 0.3}
Best F1 score found:  0.531784973226636
Test set F1 score:  0.5176470588235295


# Cat with Text

In [23]:
param_grid = {
    'classifier__learning_rate': [0.01, 0.1, 0.3],
    'classifier__l2_leaf_reg': [1, 5 ,10],
}

grid_search = GridSearchCV(estimator=baseline_model, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2, scoring='f1_macro')

grid_search.fit(X_train_combined, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_cb = grid_search.best_estimator_
y_pred = best_cb.predict(X_test_combined)
print("Test set F1 score: ", f1_score(y_test, y_pred, average='macro'))

Fitting 5 folds for each of 9 candidates, totalling 45 fits
0:	learn: 0.6466250	total: 19.2ms	remaining: 172ms
1:	learn: 0.6062006	total: 33.7ms	remaining: 135ms
2:	learn: 0.5695938	total: 47.8ms	remaining: 111ms
3:	learn: 0.5469436	total: 61.6ms	remaining: 92.4ms
4:	learn: 0.5226579	total: 75.5ms	remaining: 75.5ms
5:	learn: 0.5017965	total: 89.6ms	remaining: 59.7ms
6:	learn: 0.4839194	total: 104ms	remaining: 44.4ms
7:	learn: 0.4685484	total: 118ms	remaining: 29.4ms
8:	learn: 0.4546082	total: 131ms	remaining: 14.6ms
9:	learn: 0.4403351	total: 146ms	remaining: 0us
Best parameters found:  {'classifier__l2_leaf_reg': 5, 'classifier__learning_rate': 0.3}
Best F1 score found:  0.584151349890729
Test set F1 score:  0.5455113192818111


# Support Vector Classifier

In [24]:
baseline_model = Pipeline([('scaler',StandardScaler()),
                           ('classifier',SVC(probability=True))])

In [25]:
scores = cross_val_score(baseline_model, X_train_nontext, y_train, cv=5)
print("baseline model score: ",np.mean(scores))

baseline model score:  0.6861937793689863


In [26]:
param_grid = {
    'classifier__C': [0.1, 1, 10, 100],
    'classifier__gamma': ['scale', 'auto'],
    'classifier__kernel': ['linear', 'poly', 'rbf', 'sigmoid']
}

grid_search = GridSearchCV(estimator=baseline_model, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2, scoring='f1_macro')

grid_search.fit(X_train_nontext, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_svc = grid_search.best_estimator_
y_pred = best_svc.predict(X_test_nontext)
print("Test set F1 score: ", f1_score(y_test, y_pred, average='macro'))

Fitting 5 folds for each of 32 candidates, totalling 160 fits
Best parameters found:  {'classifier__C': 100, 'classifier__gamma': 'auto', 'classifier__kernel': 'rbf'}
Best F1 score found:  0.5496265156257251
Test set F1 score:  0.5490824534942182


# SVC with Text

In [27]:
param_grid = {
    'classifier__C': [0.01 ,0.1, 1],
    'classifier__gamma': ['scale'],
    'classifier__kernel': ['rbf', 'sigmoid']
}

grid_search = GridSearchCV(estimator=baseline_model, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2, scoring='f1_macro')

grid_search.fit(X_train_combined, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_svc = grid_search.best_estimator_
y_pred = best_svc.predict(X_test_combined)
print("Test set F1 score: ", f1_score(y_test, y_pred, average='macro'))

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best parameters found:  {'classifier__C': 1, 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf'}
Best F1 score found:  0.6524896823684225
Test set F1 score:  0.6091452037382568


# XGBoost

In [28]:
baseline_model = Pipeline([('scaler',StandardScaler()),
                           ('classifier',XGBClassifier())])

In [29]:
le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [30]:
scores = cross_val_score(baseline_model, X_train_nontext, y_train, cv=5)
print("baseline model score: ",np.mean(scores))

baseline model score:  0.6599369732229432


In [31]:
param_grid = {
    'n_estimators': [25, 50, 100],
    'max_depth': [5, 7, 9],
    'learning_rate': [1e-3, 1e-2, 1e-1],
    'subsample': [0.4, 0.6, 0.8],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

xgb_model = XGBClassifier(eval_metric='logloss')

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5,
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X_train_nontext, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_xgb = grid_search.best_estimator_
y_pred = best_xgb.predict(X_test_nontext)
print("Test set F1 score: ", f1_score(y_test, y_pred, average='macro'))

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters found:  {'colsample_bytree': 0.6, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.6}
Best F1 score found:  0.6215436610308532
Test set F1 score:  0.6232185362620145


# XGBoost With Text

In [32]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 7, 9],
    'learning_rate': [1e-3, 1e-2, 1e-1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

xgb_model = XGBClassifier(eval_metric='logloss', random_state=0)

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5,
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X_train_combined, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_xgb = grid_search.best_estimator_
y_pred = best_xgb.predict(X_test_combined)
print("Test set F1 score: ", f1_score(y_test, y_pred, average='macro'))

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters found:  {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.6}
Best F1 score found:  0.6989907133546746
Test set F1 score:  0.6394978535381524


In [33]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.73      0.88      0.80       269
           1       0.62      0.39      0.48       141

    accuracy                           0.71       410
   macro avg       0.68      0.63      0.64       410
weighted avg       0.70      0.71      0.69       410

